[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.4_nvidia_dynamo/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/06_engines/05.4_nvidia_dynamo/lab.ipynb)

# Lab 5.4: NVIDIA Dynamo Analytics

Analytical experiments exploring disaggregated P/D serving economics,
NIXL transfer overhead, KV-aware routing benefits, and pool scaling decisions.

In [ ]:
# ============================================================
# SETUP: Import libraries and define model parameters
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# --- Model parameters (Llama 3.1 70B) ---
NUM_LAYERS = 80          # transformer layers
NUM_KV_HEADS = 8         # GQA with 8 KV heads
HEAD_DIM = 128           # dimension per head
DTYPE_BYTES = 2          # FP16 = 2 bytes per element

# --- Hardware parameters ---
NVLINK_BW_GBS = 900      # NVLink bandwidth GB/s
IB_RDMA_BW_GBS = 50      # InfiniBand GPUDirect RDMA GB/s
PCIE_BW_GBS = 64         # PCIe Gen5 bandwidth GB/s
GPU_MEM_BW_GBS = 3350    # H100 HBM3 bandwidth GB/s

# --- Serving parameters ---
PREFILL_TOKS_PER_SEC = 50000  # prefill throughput (tokens/s per node)
DECODE_TOKS_PER_SEC = 2000    # decode throughput (tokens/s per sequence)

print("Setup complete. Model: Llama 3.1 70B (GQA, 8 KV heads)")

## Experiment 1: KV Cache Size vs Context Length

Compute KV cache size as a function of sequence length to understand
NIXL transfer costs at different conversation depths.

In [ ]:
# ============================================================
# EXPERIMENT 1: KV cache size scaling with sequence length
# ============================================================
def kv_cache_size_bytes(seq_len):
    """Compute KV cache size for one sequence.
    Formula: 2 * layers * kv_heads * head_dim * seq_len * dtype_bytes
    Factor of 2 accounts for both K and V tensors."""
    # Return the computed result
    return 2 * NUM_LAYERS * NUM_KV_HEADS * HEAD_DIM * seq_len * DTYPE_BYTES

# Sweep context lengths from 512 to 128K tokens
seq_lengths = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
# Compute KV size in GB for each length
kv_sizes_gb = np.array([kv_cache_size_bytes(s) / 1e9 for s in seq_lengths])

# Configure plot element
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
# Bar chart showing KV cache growth
bars = ax.bar(range(len(seq_lengths)), kv_sizes_gb, color='#6366f1', alpha=0.8)
# Configure plot element
ax.set_xticks(range(len(seq_lengths)))
# Configure plot element
ax.set_xticklabels([f'{s//1024}K' if s >= 1024 else str(s) for s in seq_lengths], fontsize=9)
# Configure plot element
ax.set_xlabel('Context Length (tokens)')
# Configure plot element
ax.set_ylabel('KV Cache Size (GB)')
# Configure plot element
ax.set_title('KV Cache Size per Sequence (Llama 70B, GQA 8 KV heads, FP16)')
# Annotate each bar with the exact value
# Iterate over each item
for bar, val in zip(bars, kv_sizes_gb):
    # Configure plot element
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f} GB', ha='center', fontsize=8)
# Configure plot element
ax.set_ylim(0, max(kv_sizes_gb) * 1.15)
# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

# Display results to user
print(f"At 32K tokens: {kv_cache_size_bytes(32768)/1e9:.2f} GB")
# Display results to user
print(f"At 128K tokens: {kv_cache_size_bytes(131072)/1e9:.2f} GB")

## Experiment 2: NIXL Transfer Time by Interconnect

Compare KV cache transfer latency across NVLink, InfiniBand RDMA,
and CPU-mediated paths. This determines whether disaggregation is practical.

In [ ]:
# ============================================================
# EXPERIMENT 2: Transfer latency comparison across interconnects
# ============================================================
def transfer_time_ms(size_gb, bw_gbs):
    """Transfer time in milliseconds given size (GB) and bandwidth (GB/s)."""
    # Return the computed result
    return (size_gb / bw_gbs) * 1000

def cpu_mediated_time_ms(size_gb):
    """CPU-mediated path: GPU->CPU (PCIe) + network (IB) + CPU->GPU (PCIe)."""
    gpu_to_cpu = size_gb / PCIE_BW_GBS * 1000  # milliseconds
    network = size_gb / IB_RDMA_BW_GBS * 1000  # same wire, but CPU copies
    cpu_to_gpu = size_gb / PCIE_BW_GBS * 1000
    # Return the computed result
    return gpu_to_cpu + network + cpu_to_gpu

# Compute transfer times for each interconnect at each context length
nvlink_ms = [transfer_time_ms(s, NVLINK_BW_GBS) for s in kv_sizes_gb]
ib_rdma_ms = [transfer_time_ms(s, IB_RDMA_BW_GBS) for s in kv_sizes_gb]
cpu_ms = [cpu_mediated_time_ms(s) for s in kv_sizes_gb]

# Configure plot element
fig_2, ax_2 = plt.subplots(1, 1, figsize=(10, 6))
x = np.arange(len(seq_lengths))
width = 0.25

# Grouped bars for each interconnect type
ax_2.bar(x - width, nvlink_ms, width, label='NVLink (900 GB/s)', color='#22c55e', alpha=0.85)
ax_2.bar(x, ib_rdma_ms, width, label='IB GPUDirect RDMA (50 GB/s)', color='#f59e0b', alpha=0.85)
ax_2.bar(x + width, cpu_ms, width, label='CPU-mediated (PCIe+IB+PCIe)', color='#ef4444', alpha=0.85)

ax_2.set_xticks(x)
ax_2.set_xticklabels([f'{s//1024}K' if s >= 1024 else str(s) for s in seq_lengths], fontsize=9)
ax_2.set_xlabel('Context Length (tokens)')
ax_2.set_ylabel('Transfer Time (ms)')
ax_2.set_title('KV Cache Transfer Latency: NIXL vs CPU-Mediated')
ax_2.legend()
ax_2.set_yscale('log')  # log scale because range spans 0.1ms to 5000ms
# Add a horizontal line at 100ms as a "budget" threshold
ax_2.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='100ms budget')
ax_2.text(0.1, 110, '100ms latency budget', fontsize=8, color='gray')
# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

# Print key data points
# Display results to user
print("\n32K context (10.7 GB):")
# Display results to user
print(f"  NVLink:      {transfer_time_ms(kv_sizes_gb[6], NVLINK_BW_GBS):.1f} ms")
# Display results to user
print(f"  IB RDMA:     {transfer_time_ms(kv_sizes_gb[6], IB_RDMA_BW_GBS):.1f} ms")
# Display results to user
print(f"  CPU-mediated: {cpu_mediated_time_ms(kv_sizes_gb[6]):.1f} ms")

## Experiment 3: Disaggregation Break-Even Analysis

At what context length does NIXL transfer overhead negate the benefits
of disaggregated serving? Compare TTFT with and without disaggregation.

In [ ]:
# ============================================================
# EXPERIMENT 3: Disaggregated vs aggregated TTFT comparison
# ============================================================

# Assumptions for aggregated serving:
# - Prefill competes with decode for compute, causing ~2x slowdown
AGGREGATED_PREFILL_PENALTY = 2.0  # prefill takes 2x longer when sharing GPU with decode

# Assumptions for disaggregated serving:
# - Dedicated prefill worker runs at full speed
# - But adds NIXL transfer overhead after prefill

context_lens = np.linspace(256, 131072, 200)  # sweep from 256 to 128K

def ttft_aggregated_ms(seq_len):
    """TTFT in aggregated mode: prefill is slower due to decode interference."""
    base_prefill_ms = (seq_len / PREFILL_TOKS_PER_SEC) * 1000
    # Return the computed result
    return base_prefill_ms * AGGREGATED_PREFILL_PENALTY

def ttft_disaggregated_ms(seq_len, interconnect_bw_gbs):
    """TTFT in disaggregated mode: fast prefill + NIXL transfer."""
    base_prefill_ms = (seq_len / PREFILL_TOKS_PER_SEC) * 1000  # no penalty
    kv_gb = kv_cache_size_bytes(int(seq_len)) / 1e9
    transfer_ms = (kv_gb / interconnect_bw_gbs) * 1000
    # Return the computed result
    return base_prefill_ms + transfer_ms

# Compute TTFT curves
ttft_agg = [ttft_aggregated_ms(s) for s in context_lens]
ttft_dis_nvlink = [ttft_disaggregated_ms(s, NVLINK_BW_GBS) for s in context_lens]
ttft_dis_ib = [ttft_disaggregated_ms(s, IB_RDMA_BW_GBS) for s in context_lens]

# Configure plot element
fig_3, ax_3 = plt.subplots(1, 1, figsize=(10, 6))
ax_3.plot(context_lens/1000, ttft_agg, 'r-', linewidth=2, label='Aggregated (2x prefill penalty)')
ax_3.plot(context_lens/1000, ttft_dis_nvlink, 'g-', linewidth=2, label='Disaggregated + NVLink')
ax_3.plot(context_lens/1000, ttft_dis_ib, color='#f59e0b', linewidth=2, label='Disaggregated + IB RDMA')

# Find crossover point for IB RDMA
# Iterate over each item
for i in range(len(context_lens)-1):
    # Conditional check
    if ttft_dis_ib[i] < ttft_agg[i] and ttft_dis_ib[i+1] >= ttft_agg[i+1]:
        ax_3.axvline(x=context_lens[i]/1000, color='gray', linestyle=':', alpha=0.5)
        ax_3.text(context_lens[i]/1000, max(ttft_agg)*0.9, f'IB crossover\n~{context_lens[i]/1000:.0f}K', fontsize=8)
        break

ax_3.set_xlabel('Context Length (K tokens)')
ax_3.set_ylabel('Time to First Token (ms)')
ax_3.set_title('TTFT: Aggregated vs Disaggregated Serving')
ax_3.legend()
ax_3.grid(True, alpha=0.3)
# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

# Display results to user
print("Key insight: Disaggregation with NVLink ALWAYS wins.")
# Display results to user
print("With IB RDMA, it wins up to the crossover point where transfer overhead dominates.")

## Experiment 4: KV Affinity Routing Savings

Quantify the latency savings from KV-aware routing in multi-turn conversations.
Without affinity, every turn re-prefills the full conversation history.

In [ ]:
# ============================================================
# EXPERIMENT 4: Multi-turn latency with and without KV affinity
# ============================================================

# Simulate a 10-turn conversation where each user message is ~200 tokens
# and each assistant response is ~500 tokens
USER_MSG_TOKENS = 200    # tokens per user message
ASSISTANT_TOKENS = 500   # tokens per assistant response
NUM_TURNS = 10           # total conversation turns
SYSTEM_PROMPT_TOKENS = 1000  # fixed system prompt

def cumulative_context_at_turn(turn):
    """Total tokens in KV cache at the start of turn N."""
    # Return the computed result
    return SYSTEM_PROMPT_TOKENS + turn * (USER_MSG_TOKENS + ASSISTANT_TOKENS)

def ttft_no_affinity_ms(turn):
    """Without affinity: must re-prefill entire conversation history each turn."""
    total_tokens = cumulative_context_at_turn(turn) + USER_MSG_TOKENS
    # Return the computed result
    return (total_tokens / PREFILL_TOKS_PER_SEC) * 1000

def ttft_with_affinity_ms(turn):
    """With affinity: only prefill the new user message (KV cache already on worker)."""
    # Return the computed result
    return (USER_MSG_TOKENS / PREFILL_TOKS_PER_SEC) * 1000

turns = np.arange(1, NUM_TURNS + 1)
ttft_no_aff = [ttft_no_affinity_ms(t) for t in turns]
ttft_with_aff = [ttft_with_affinity_ms(t) for t in turns]
# Cumulative savings in ms
savings_ms = [ttft_no_aff[i] - ttft_with_aff[i] for i in range(len(turns))]

# Configure plot element
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: TTFT per turn
# Configure plot element
ax1.plot(turns, ttft_no_aff, 'r-o', label='No affinity (re-prefill all)')
# Configure plot element
ax1.plot(turns, ttft_with_aff, 'g-o', label='KV affinity (prefill new msg only)')
# Configure plot element
ax1.fill_between(turns, ttft_with_aff, ttft_no_aff, alpha=0.15, color='green')
# Configure plot element
ax1.set_xlabel('Conversation Turn')
# Configure plot element
ax1.set_ylabel('TTFT (ms)')
# Configure plot element
ax1.set_title('TTFT per Turn: Affinity vs No Affinity')
# Configure plot element
ax1.legend()
# Configure plot element
ax1.grid(True, alpha=0.3)

# Right: Cumulative context growth
context_at_turn = [cumulative_context_at_turn(t) for t in turns]
ax2.bar(turns, [c/1000 for c in context_at_turn], color='#6366f1', alpha=0.7)
# Configure plot element
ax2.set_xlabel('Conversation Turn')
# Configure plot element
ax2.set_ylabel('Accumulated Context (K tokens)')
# Configure plot element
ax2.set_title('KV Cache Growth Over Conversation')
# Configure plot element
ax2.grid(True, alpha=0.3)

# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

# Display results to user
print(f"\nAt turn 10: context = {cumulative_context_at_turn(10):,} tokens")
# Display results to user
print(f"  Without affinity TTFT: {ttft_no_affinity_ms(10):.1f} ms")
# Display results to user
print(f"  With affinity TTFT:    {ttft_with_affinity_ms(10):.1f} ms")
# Display results to user
print(f"  Savings per request:   {ttft_no_affinity_ms(10) - ttft_with_affinity_ms(10):.1f} ms")

## Experiment 5: Pool Sizing Economics

Given a target SLO (TTFT < 500ms, ITL < 30ms), compute the optimal
prefill:decode worker ratio for different traffic patterns.

In [ ]:
# ============================================================
# EXPERIMENT 5: Optimal P:D ratio under different workloads
# ============================================================

# Traffic patterns: (avg_prompt_len, avg_output_len, description)
WORKLOADS = [
    (500, 100, 'Chatbot (short prompts, short replies)'),
    (4000, 500, 'Code assistant (long context, medium output)'),
    (16000, 2000, 'Document QA (very long context, long output)'),
    (1000, 4000, 'Creative writing (medium prompt, very long output)'),
]

# Assume 8 total GPU nodes to allocate between P and D pools
TOTAL_NODES = 8
# Each prefill node handles PREFILL_TOKS_PER_SEC tokens of prefill
# Each decode node handles max 256 concurrent sequences
DECODE_BATCH_SIZE = 256

def compute_optimal_ratio(prompt_len, output_len, total_nodes):
    """Find P:D ratio that balances TTFT and ITL SLOs.
    Returns (prefill_nodes, decode_nodes, ttft_ms, itl_ms)."""
    best = None
    # Try each possible split
    # Iterate over each item
    for p_nodes in range(1, total_nodes):
        d_nodes = total_nodes - p_nodes
        # TTFT: prompt_len / (prefill_throughput * p_nodes)
        ttft = (prompt_len / (PREFILL_TOKS_PER_SEC * p_nodes)) * 1000
        # ITL: inversely proportional to decode bandwidth per node
        # More decode nodes = more concurrent sequences = lower ITL
        itl = 1000 / (DECODE_TOKS_PER_SEC)  # ms per token (constant per seq)
        # Score: weighted sum of SLO violations
        score = max(0, ttft - 500) * 2 + max(0, itl - 30) * 5
        # Conditional check
        if best is None or score < best[0]:
            best = (score, p_nodes, d_nodes, ttft, itl)
    # Return the computed result
    return best[1], best[2], best[3], best[4]

# Configure plot element
fig_5, ax_5 = plt.subplots(1, 1, figsize=(10, 6))
colors = ['#6366f1', '#22c55e', '#f59e0b', '#ef4444']

# Iterate over each item
for i, (prompt_len, output_len, desc) in enumerate(WORKLOADS):
    # Compute TTFT for each possible prefill allocation
    p_range = range(1, TOTAL_NODES)
    ttfts = [(prompt_len / (PREFILL_TOKS_PER_SEC * p)) * 1000 for p in p_range]
    ax_5.plot(list(p_range), ttfts, '-o', color=colors[i], label=desc, linewidth=2)

# SLO threshold line
ax_5.axhline(y=500, color='red', linestyle='--', alpha=0.6, linewidth=1.5)
ax_5.text(1.1, 520, 'TTFT SLO = 500ms', color='red', fontsize=9)

ax_5.set_xlabel('Prefill Workers (out of 8 total)')
ax_5.set_ylabel('TTFT (ms)')
ax_5.set_title('TTFT vs Prefill Pool Size by Workload Type')
ax_5.legend(loc='upper right', fontsize=9)
ax_5.set_xticks(range(1, TOTAL_NODES))
# Annotate decode workers on secondary x-axis
ax2 = ax_5.twiny()
# Configure plot element
ax2.set_xlim(ax_5.get_xlim())
# Configure plot element
ax2.set_xticks(range(1, TOTAL_NODES))
# Configure plot element
ax2.set_xticklabels([f'{TOTAL_NODES-p}D' for p in range(1, TOTAL_NODES)])
# Configure plot element
ax2.set_xlabel('Decode Workers')
ax_5.grid(True, alpha=0.3)
# Configure plot element
plt.tight_layout()
# Configure plot element
plt.show()

# Print optimal ratios
# Display results to user
print("\nOptimal P:D ratios (8 nodes total, TTFT SLO 500ms):")
# Iterate over each item
for prompt_len, output_len, desc in WORKLOADS:
    p, d, ttft, itl = compute_optimal_ratio(prompt_len, output_len, TOTAL_NODES)
    # Display results to user
    print(f"  {desc}: {p}P:{d}D (TTFT={ttft:.0f}ms)")

## Key Takeaways

1. **KV cache grows linearly** with context length. At 32K tokens, a single Llama 70B sequence needs ~10.7 GB of KV cache.
2. **NVLink makes disaggregation free.** At 900 GB/s, even 128K-context transfers complete in ~48ms.
3. **IB RDMA is practical up to ~32K context.** Beyond that, transfer overhead starts competing with prefill savings.
4. **KV affinity is critical for multi-turn.** Without it, TTFT grows linearly with conversation length as each turn re-prefills everything.
5. **Pool ratios depend on workload.** Document QA (long prompts) needs more prefill nodes; creative writing (long outputs) needs more decode nodes.